# Pipeline Control Utilities (utils.ipynb)
Provides reusable functions for metadata lookup, watermark tracking, and status updating in Medallion Lakehouse pipelines.

In [ ]:
# 1. Initialize SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
%run ./logger

In [ ]:
# 2. Initialize Centralized Logger for Utilities
logger = get_task_logger(notebook_name_override="pipeline_utils")

In [ ]:
# 3. Extract Metadata by Table ID or Table Name
def extract_table_metadata(table_id_or_name):
    """
    Fetches pipeline metadata from spark_training.metadata_schema.metadata_table.
    Accepts either an integer table_id (e.g. 101) or a string table_name (e.g. 'countries').
    Returns a dictionary of metadata attributes for clean, easy access.
    """
    if isinstance(table_id_or_name, int) or (isinstance(table_id_or_name, str) and table_id_or_name.isdigit()):
        filter_clause = f"table_id = {int(table_id_or_name)}"
    else:
        filter_clause = f"lower(source_table_name) = lower('{table_id_or_name}')"

    query = f"""
        SELECT *
        FROM spark_training.metadata_schema.metadata_table
        WHERE {filter_clause}
    """
    df = spark.sql(query)
    rows = df.collect()
    if not rows:
        raise ValueError(f"No metadata found matching: '{table_id_or_name}' in spark_training.metadata_schema.metadata_table")
    return rows[0].asDict()

In [ ]:
# 4. Get All Active Tables for a Target Layer
def get_active_tables(target_layer: str = "bronze"):
    """
    Retrieves all active tables configured for a specific target layer (e.g. 'bronze', 'silver').
    Returns a list of dictionaries, ordered by table_id.
    """
    query = f"""
        SELECT *
        FROM spark_training.metadata_schema.metadata_table
        WHERE is_active = 'true' AND lower(target_name) = lower('{target_layer}')
        ORDER BY table_id
    """
    df = spark.sql(query)
    return [row.asDict() for row in df.collect()]

In [ ]:
# 5. Update Last Load Date (Watermark)
def update_last_load_date(table_id: int, last_load_date):
    """
    Updates the last_load_date watermark and updated_on timestamp for a specific table.
    """
    try:
        query = f"""
            UPDATE spark_training.metadata_schema.metadata_table
            SET last_load_date = '{last_load_date}',
                updated_on = CURRENT_DATE(),
                updated_by = 'sham'
            WHERE table_id = {table_id}
        """
        spark.sql(query)
        logger.info(f"Updated last_load_date to '{last_load_date}' for table_id {table_id}")
        return f"Successfully updated last_load_date for table_id {table_id}"
    except Exception as e:
        logger.error(f"Failed to update last_load_date for table_id {table_id}: {e}")
        raise e

In [ ]:
# 6. Update Last Run Status
def update_last_load_status(table_id: int, status: str):
    """
    Updates the last_run_status ('SUCCESS', 'FAILED', 'RUNNING') and updated_on for a specific table.
    """
    try:
        query = f"""
            UPDATE spark_training.metadata_schema.metadata_table
            SET last_run_status = '{status}',
                updated_on = CURRENT_DATE(),
                updated_by = 'sham'
            WHERE table_id = {table_id}
        """
        spark.sql(query)
        logger.info(f"Updated last_run_status to '{status}' for table_id {table_id}")
        return f"Successfully updated last_run_status for table_id {table_id}"
    except Exception as e:
        logger.error(f"Failed to update last_run_status for table_id {table_id}: {e}")
        raise e

In [ ]:
# 7. Interactive Test & Verification (uncomment to test directly in notebook)
# meta = extract_table_metadata(101)
# print(f"Test Metadata: ID {meta['table_id']} -> {meta['source_table_name']} (Pushdown: {meta['query'][:40]}...)")
# print(f"Active Bronze Tables Count: {len(get_active_tables('bronze'))}")